# Multiple Linear Example

In [194]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import seaborn as sns

In [195]:
# Univariate = SLP
# Multivariate  = MLP
DATASET = "50_Startups.csv"

In [196]:
df = pd.read_csv(DATASET)

In [197]:
df.head()

,RnD,Administration,Marketing,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [198]:
# EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   RnD             50 non-null     float64
 1   Administration  50 non-null     float64
 2   Marketing       50 non-null     float64
 3   State           50 non-null     object 
 4   Profit          50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [199]:
# convert `state` column to numeric
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

df['State'] = encoder.fit_transform(df['State'])
df.info()
# df = df.drop('Satate',axis=1)
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   RnD             50 non-null     float64
 1   Administration  50 non-null     float64
 2   Marketing       50 non-null     float64
 3   State           50 non-null     int64  
 4   Profit          50 non-null     float64
dtypes: float64(4), int64(1)
memory usage: 2.1 KB


,RnD,Administration,Marketing,State,Profit
0,165349.20,136897.80,471784.10,2,192261.83
1,162597.70,151377.59,443898.53,0,191792.06
2,153441.51,101145.55,407934.54,1,191050.39
3,144372.41,118671.85,383199.62,2,182901.99
4,142107.34,91391.77,366168.42,1,166187.94


# data preparation


In [200]:
x = df.drop('Profit',axis=1)
y = df['Profit']

print(x.ndim,y.ndim)

2 1


In [201]:
# split data to train and test
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.7,random_state=42)

In [202]:
# We use .values because PyTorch tensors
# cannot directly work with pandas DataFrames and Series in all cases.
# We first convert the pandas data into a NumPy array,
# then convert that array into a PyTorch tensor.

### Convert data to tensor

In [203]:
# coverting 
x_train = torch.FloatTensor(x_train.values)
x_test = torch.FloatTensor(x_test.values)

# since y_train and y_test are single dimension we need to convert them to 2D
y_train = torch.FloatTensor(y_train.values).view(-1,1)
y_test = torch.FloatTensor(y_test.values).view(-1,1)

In [204]:
# here x_train.ndim, y_train.ndim are not same so we need to make them same 
x_train.ndim, y_train.ndim

(2, 2)

# Build model

#### define hyperparameters


In [205]:
# epochs
epochs = 100


In [206]:
x_train.shape

torch.Size([35, 4])

In [207]:
# create model
# use torch module called torch.nn.Sequantial() -->  sequence of layers
# fisrt layer is our Linear layer with input_fe = 4 and o/ = 8 
# add activation function on layer
# sequnatial is containor to ccontain all fully connceted layer
# means all nurons form  previous layer will be connceted to all the nurons of next layer
#first layer always INPUT LAYER and Last always OUTPUT LAYER
model = torch.nn.Sequential(

    # input layer connceting to hidden layer with 8 nuron
    # Here first input layer come from no. of feature and it create 1st hidden layer  
    # we can say it will create input layer + 1 hidden layer
    # by default it is going to add 2 layers
    torch.nn.Linear(in_features=x_train.shape[1],out_features=8),
    # set activation function at hidden layer as ReLU
    torch.nn.ReLU(),

    # then input for next hidden layer must same as output of its preivious layer
    # add second  Hidden layer with 4 nurons
    torch.nn.Linear(in_features=8,out_features=4),
    # activation functin
    torch.nn.ReLU(),

    # output layer
    torch.nn.Linear(in_features=4,out_features=1)
)

| Problem Type                   | Output Activation                                             |
| ------------------------------ | ------------------------------------------------------------- |
| **Regression**                 | ❌ No activation                                               |
| **Binary Classification**      | ✅ Sigmoid                                                     |
| **Multi-class Classification** | ❌ Usually no activation in the model (use `CrossEntropyLoss`) |
| **Multi-label Classification** | ✅ Sigmoid                                                     |


In [208]:
# get model info
model


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

# training Loop

#### hyperparameter

In [209]:
# #epochs
epochs = 10000

# loss function
loss_function = torch.nn.MSELoss()

# learning rate 
lr =0.001

#optimizer
optimizer = torch.optim.SGD(model.parameters(),lr =lr)

#### trainning loop


In [210]:
losses = []
for epoch in range(epochs):

    # enable the model training mode
    # It does not train model
    # it just flag it not do : just enable flag we are entring in training mode
    model.train()

    # trainning model
    prediction = model(x_train)

    # collect all losses
    loss = loss_function(y_train,prediction)

    # zero out gradients 
    optimizer.zero_grad() 

    # calculte gradients
    loss.backward()

    # optimiz the model parameter
    optimizer.step()

    #collect the loss
    losses.append(loss)

    # print the training progress every after 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"epoch : {epoch} , loss : {loss}") 

epoch : 99 , loss : nan
epoch : 199 , loss : nan


epoch : 299 , loss : nan
epoch : 399 , loss : nan
epoch : 499 , loss : nan
epoch : 599 , loss : nan
epoch : 699 , loss : nan
epoch : 799 , loss : nan
epoch : 899 , loss : nan
epoch : 999 , loss : nan
epoch : 1099 , loss : nan
epoch : 1199 , loss : nan
epoch : 1299 , loss : nan
epoch : 1399 , loss : nan
epoch : 1499 , loss : nan
epoch : 1599 , loss : nan
epoch : 1699 , loss : nan
epoch : 1799 , loss : nan
epoch : 1899 , loss : nan
epoch : 1999 , loss : nan
epoch : 2099 , loss : nan
epoch : 2199 , loss : nan
epoch : 2299 , loss : nan
epoch : 2399 , loss : nan
epoch : 2499 , loss : nan
epoch : 2599 , loss : nan
epoch : 2699 , loss : nan
epoch : 2799 , loss : nan
epoch : 2899 , loss : nan
epoch : 2999 , loss : nan
epoch : 3099 , loss : nan
epoch : 3199 , loss : nan
epoch : 3299 , loss : nan
epoch : 3399 , loss : nan
epoch : 3499 , loss : nan
epoch : 3599 , loss : nan
epoch : 3699 , loss : nan
epoch : 3799 , loss : nan
epoch : 3899 , loss : nan
epoch : 3999 , loss : nan
epoch : 4099 , loss 

### Evaluating model

In [211]:
# evaluate model
# it does not evaluate model 
# here model do not learn any paarameter after setting this  
# enable model evaluation mode
model.eval()

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [212]:
# don not calculate any gradience
with torch.no_grad():

    # collect loss 
    prediction= model(x_test)

    # calculate evaluation loss
    evalution_loss = loss_function(y_test,prediction)

print(f"evaluation MSE = {evalution_loss}")

evaluation MSE = nan


In [213]:
list(model.parameters())

[Parameter containing:
 tensor([[    nan,     nan,     nan,     nan],
         [ 0.1450, -0.1343, -0.3975, -0.3355],
         [ 0.3093, -0.3135, -0.0564,  0.3283],
         [-0.2378, -0.1765, -0.3342,  0.0167],
         [    nan,     nan,     nan,     nan],
         [ 0.1494, -0.1185, -0.0597,  0.3353],
         [-0.4839, -0.2559, -0.0971, -0.3253],
         [    nan,     nan,     nan,     nan]], requires_grad=True),
 Parameter containing:
 tensor([    nan,  0.4171,  0.1552,  0.4572,     nan, -0.2051, -0.0531,     nan],
        requires_grad=True),
 Parameter containing:
 tensor([[nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan]], requires_grad=True),
 Parameter containing:
 tensor([nan, nan, nan, nan], requires_grad=True),
 Parameter containing:
 tensor([[nan, nan, nan, nan]], requires_grad=True),
 Parameter containing:
 tensor([nan], requires

# Model Inference

In [214]:
# predict the profit 
with torch.no_grad():
    model(torch.tensor([[	165349.20	,136897.80	,471784.10	,2]]))

In [215]:
df.head()

,RnD,Administration,Marketing,State,Profit
0,165349.20,136897.80,471784.10,2,192261.83
1,162597.70,151377.59,443898.53,0,191792.06
2,153441.51,101145.55,407934.54,1,191050.39
3,144372.41,118671.85,383199.62,2,182901.99
4,142107.34,91391.77,366168.42,1,166187.94


# saving model

In [ ]:
torch.save('mymodel.h5')